In [1]:
!pip install --upgrade \
  --index-url https://download.pytorch.org/whl/cu118 \
  torch==2.1.0+cu118 \
  torchvision==0.16.0+cu118 \
  torchaudio==2.1.0+cu118


Looking in indexes: https://download.pytorch.org/whl/cu118


In [2]:
!pip install \
  transformers datasets peft accelerate huggingface-hub sentencepiece protobuf bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 10.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of fsspec[http] to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.9/73.9 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 73.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 367.1/367.1 kB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.8/558.8 kB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from huggingface_hub import login

login(token="")  # replace with your actual token


In [4]:
# 2) Load & tokenize your Hub dataset
from datasets import load_dataset
from transformers import AutoTokenizer

# 1) load your dataset
dataset = load_dataset("mdot77/sp500p", split="train")

# 2) init tokenizer and ensure it has a pad_token
base_model_id = "meta-llama/Llama-2-7b-chat-hf"
tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    use_fast=True,
)
# assign EOS as PAD if none is defined
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def preprocess(batch):
    # join prompt + response into one sequence
    examples = [p + r for p, r in zip(batch["prompt"], batch["response"])]
    toks = tokenizer(
        examples,
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    # causal‐LM: predict the entire sequence
    toks["labels"] = toks["input_ids"].copy()
    return toks

# 3) tokenize and remove original columns
tokenized = dataset.map(
    preprocess,
    batched=True,
    remove_columns=["prompt", "response"],
)

# 4) set to PyTorch tensors
tokenized.set_format("torch")


README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1509 [00:00<?, ? examples/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Map:   0%|          | 0/1509 [00:00<?, ? examples/s]

In [5]:
# 3) Load base LLaMA + your FinGPT LoRA adapter
from transformers import AutoModelForCausalLM
from peft import PeftModel, prepare_model_for_kbit_training

# 3a) Load 8‑bit for VRAM savings (optional)
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    load_in_8bit=True,
)
# 1) Turn off KV‑cache (so it won’t auto‑disable later)
model.config.use_cache = False

# 2) Enable gradient checkpointing (optional but saves memory)
model.gradient_checkpointing_enable()

# 3) Allow gradients to flow into the input & adapter layers
model.enable_input_require_grads()
model = prepare_model_for_kbit_training(model)
# 3b) Attach your FinGPT LoRA weights
peft_model_id = "FinGPT/fingpt-forecaster_dow30_llama2-7b_lora"
model = PeftModel.from_pretrained(model, peft_model_id, device_map="auto")


/usr/local/lib/python3.10/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/usr/local/lib/python3.10/dist-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/528 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/40.0M [00:00<?, ?B/s]

In [6]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

# 1) Data collator for causal LM (builds labels automatically)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 2) TrainingArguments: add label_names so Trainer knows to use your "labels" field
training_args = TrainingArguments(
    output_dir                  = "./llama2-7b-finetuned",
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 8,
    learning_rate               = 2e-5,
    fp16                        = False,       # ← disable mix‑precision
    bf16                        = False,       # ← make sure BF16 is off too
    num_train_epochs            = 4,
    logging_steps               = 50,
    save_steps                  = 500,
    push_to_hub                 = True,
    hub_model_id                = "mdot77/fingpt-llama2-7b-forecaster-finetuned",
    label_names                 = ["labels"],
)
# 3) Instantiate Trainer without any custom compute_loss or subclassing
trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = tokenized,
    data_collator = data_collator,
    tokenizer     = tokenizer,
)

# 4) Run training
trainer.train()
trainer.push_to_hub()


/tmp/ipykernel_466/1193665697.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.10/dist-packages/transformers/utils/generic.py:278: FutureWarning: The input object of type 'Tensor' is an array-like implementing one of the corresponding protocols (`__array__`, `__array_interface__` or `__array_struct__`); but not a sequence (or 0-D). In the future, this object will be coerced as if it was first converted using `np.array(obj)`. To retain the old behaviour, you have to either modify the type 'Tensor', or assign to an empty array created with `np.empty(correct_shape, dtype=object)`.
  arr = np.array(obj)
/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=Fa

Step,Training Loss
50,19.628400
100,19.627200
150,19.623800


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  ...ama2-7b-finetuned/training_args.bin: 100%|##########| 5.78kB / 5.78kB            

  ...llama2-7b-finetuned/tokenizer.model: 100%|##########|  500kB /  500kB            

  ...finetuned/adapter_model.safetensors:  63%|######2   | 50.3MB / 80.0MB            

CommitInfo(commit_url='https://huggingface.co/mdot77/fingpt-llama2-7b-forecaster-finetuned/commit/010157bb70243b0cecb57306bc820e54a06fd624', commit_message='End of training', commit_description='', oid='010157bb70243b0cecb57306bc820e54a06fd624', pr_url=None, repo_url=RepoUrl('https://huggingface.co/mdot77/fingpt-llama2-7b-forecaster-finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='mdot77/fingpt-llama2-7b-forecaster-finetuned'), pr_revision=None, pr_num=None)